# TWA-FLY Phase A - CIFAR-100 train-only pilot

This notebook tests paired raw/WTA analytic consistency without opening CIFAR-100 test features. It reuses the frozen ViT and WTA caches from earlier phases. Run cells in order; send back the final ZIP whether the gate passes or fails.

In [ ]:
# === Edit paths only; do not edit the locked experiment hyperparameters ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'feature/twa-fly'
WORK_DIR = '/content/SOHO-CL'
DRIVE_ROOT = '/content/drive/MyDrive/T-SOHO'
DRIVE_FEATURE_CACHE = f'{DRIVE_ROOT}/tsoho_cifar100_cache'
DRIVE_WTA_CODE_CACHE = f'{DRIVE_ROOT}/zi_soho_wta_h10000_seed1993'
FEATURE_CACHE_DIR = '/content/tsoho_cifar100_cache'
WTA_CODE_CACHE_DIR = '/content/twa_fly_wta_h10000_seed1993'
OUTPUT_DIR = f'{DRIVE_ROOT}/twa_fly_phasea_outputs'
CHECKPOINT_SOURCE = 'huggingface'  # used only if feature extraction is necessary
DRIVE_CHECKPOINT_PATH = f'{DRIVE_ROOT}/model.safetensors'
BATCH_SIZE = 128
CHECKPOINT_SIZE = 346284714
CHECKPOINT_SHA256 = '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'

In [ ]:
# Runtime and repository setup.
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
os.chdir('/content')
shutil.rmtree(WORK_DIR, ignore_errors=True)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_GIT_URL, WORK_DIR], check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt', 'kagglehub', 'huggingface_hub'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('repo commit:', commit)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Restore feature/WTA caches from Drive with byte progress.
from tqdm.auto import tqdm
def copy_file_progress(source, destination):
    source, destination = Path(source), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with source.open('rb') as src, destination.open('wb') as dst, tqdm(total=source.stat().st_size, unit='B', unit_scale=True, desc=source.name) as bar:
        while True:
            block = src.read(8 * 1024 * 1024)
            if not block: break
            dst.write(block); bar.update(len(block))
def restore_cache(source_dir, destination_dir, required):
    source, destination = Path(source_dir), Path(destination_dir)
    if not all((source / name).is_file() for name in required):
        print('Drive cache unavailable:', source); return False
    shutil.rmtree(destination, ignore_errors=True); destination.mkdir(parents=True)
    for name in required: copy_file_progress(source / name, destination / name)
    print('Restored:', destination); return True
feature_restored = restore_cache(DRIVE_FEATURE_CACHE, FEATURE_CACHE_DIR, ('metadata.json','train.pt','test.pt'))
code_restored = restore_cache(DRIVE_WTA_CODE_CACHE, WTA_CODE_CACHE_DIR, ('metadata.json','train_codes.pt'))
print({'feature_cache_restored': feature_restored, 'wta_cache_restored': code_restored})

In [ ]:
# Extract frozen features only when the complete Drive cache was absent. Live tqdm shows progress.
required = ('metadata.json','train.pt','test.pt')
local_cache = Path(FEATURE_CACHE_DIR)
if not all((local_cache / name).is_file() for name in required):
    shutil.rmtree(local_cache, ignore_errors=True)
    if CHECKPOINT_SOURCE == 'google_drive':
        CHECKPOINT_PATH = DRIVE_CHECKPOINT_PATH
    else:
        from huggingface_hub import hf_hub_download
        CHECKPOINT_PATH = hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', filename='model.safetensors')
    import kagglehub
    downloaded = Path(kagglehub.dataset_download('zaphat206/cifar-100'))
    candidates = [downloaded, *downloaded.rglob('cifar-100')]
    cifar = next(path for path in candidates if (path/'train').is_file() and (path/'test').is_file() and (path/'meta').is_file())
    command = [sys.executable, '-u', 'tools/experiment_runner.py', '--extract-features-only', '--root', str(cifar), '--backbone-checkpoint', CHECKPOINT_PATH, '--backbone-checkpoint-size', str(CHECKPOINT_SIZE), '--backbone-checkpoint-sha256', CHECKPOINT_SHA256, '--feature-cache-dir', FEATURE_CACHE_DIR, '--output-dir', '/content/twa_extract', '--dataset', 'CIFAR-100', '--model-name', 'vit_base_patch16_224', '--data-augmentation', 'vit', '--seed', '1993', '--num-classes', '100', '--num-tasks', '10', '--device', 'cuda', '--batch-size', str(BATCH_SIZE), '--num-workers', '2']
    print('Extracting frozen ViT features; follow the live batch progress below.', flush=True)
    subprocess.run(command, check=True)
    shutil.rmtree(DRIVE_FEATURE_CACHE, ignore_errors=True)
    shutil.copytree(local_cache, DRIVE_FEATURE_CACHE)
    print('Saved feature cache to Drive:', DRIVE_FEATURE_CACHE)
else:
    print('Using restored feature cache; extraction skipped.')

In [ ]:
# Local correctness gate. Do not continue on failure.
command = [sys.executable, '-m', 'pytest', '-q', 'tests/test_twa_fly_math.py', 'tests/test_twa_fly_learner.py', 'tests/test_twa_fly_pilot.py']
completed = subprocess.run(command)
assert completed.returncode == 0, 'TWA-FLY tests failed; stop and send the traceback.'
print('TWA-FLY correctness gate: PASS')

In [ ]:
# Locked train-only pilot. One TASK line appears after each stage's analytic solves.
test_path = Path(FEATURE_CACHE_DIR, 'test.pt')
locked_test_path = Path(FEATURE_CACHE_DIR, 'test.locked.pt')
if test_path.is_file(): test_path.replace(locked_test_path)
assert locked_test_path.is_file() and not test_path.exists(), 'Could not physically hide test.pt'
command = [sys.executable, '-u', 'tools/twa_fly_pilot.py', '--config', 'configs/twa_fly_cifar100_train_only.json', '--feature-cache-dir', FEATURE_CACHE_DIR, '--code-cache-dir', WTA_CODE_CACHE_DIR, '--output-dir', OUTPUT_DIR, '--device', 'cuda', '--require-test-hidden']
print('Starting TWA-FLY Phase A (3 rho values x 3 controls, shared statistics/factors).', flush=True)
print('Silent periods are 10000D Cholesky/alternating solves; wait for TASK i/10.', flush=True)
try:
    completed = subprocess.run(command)
    assert completed.returncode == 0, 'Pilot failed; send the complete traceback without changing config.'
finally:
    if locked_test_path.is_file() and not test_path.exists(): locked_test_path.replace(test_path)
assert Path(OUTPUT_DIR, 'selection.json').is_file()
if not code_restored:
    shutil.rmtree(DRIVE_WTA_CODE_CACHE, ignore_errors=True)
    shutil.copytree(WTA_CODE_CACHE_DIR, DRIVE_WTA_CODE_CACHE)
    print('Saved reusable WTA experiment cache to Drive:', DRIVE_WTA_CODE_CACHE)
print('TWA-FLY train-only pilot: COMPLETE')

In [ ]:
# Compact result and evidence download.
import pandas as pd
payload = json.loads(Path(OUTPUT_DIR, 'selection.json').read_text())
rows = [{k: row.get(k) for k in ('method','rho','validation_average_accuracy','persistent_state_bytes')} for row in payload['candidates']]
display(pd.DataFrame(rows).sort_values(['method','rho'], na_position='first'))
print(json.dumps(payload['gate'], indent=2))
archive_base = '/content/twa_fly_phasea_train_only'
shutil.make_archive(archive_base, 'zip', OUTPUT_DIR)
from google.colab import files
files.download(archive_base + '.zip')
print('Send back twa_fly_phasea_train_only.zip. Do not evaluate test in this phase.')